In [3]:
import random
import tkinter as tk

# Sorting Algorithm Generators
def bubble_sort_generator(arr):
    a = arr[:]
    n = len(a)
    for i in range(n):
        for j in range(n - i - 1):
            highlight = [j, j + 1]
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
            yield a[:], highlight

def selection_sort_generator(arr):
    a = arr[:]
    n = len(a)
    for i in range(n):
        min_idx = i
        for j in range(i + 1, n):
            highlight = [min_idx, j]
            if a[j] < a[min_idx]:
                min_idx = j
            yield a[:], highlight
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            yield a[:], [i, min_idx]

def insertion_sort_generator(arr):
    a = arr[:]
    n = len(a)
    for i in range(1, n):
        j = i
        while j > 0 and a[j] < a[j - 1]:
            highlight = [j, j - 1]
            a[j], a[j - 1] = a[j - 1], a[j]
            yield a[:], highlight
            j -= 1

def merge_sort_generator(arr):
    a = arr[:]
    n = len(a)

    def _merge_sort(l, r):
        if r - l > 1:
            m = (l + r) // 2
            yield from _merge_sort(l, m)
            yield from _merge_sort(m, r)
            left, right = a[l:m], a[m:r]
            i = j = 0
            k = l
            while i < len(left) and j < len(right):
                highlight = list(range(l, r))
                if left[i] <= right[j]:
                    a[k] = left[i]
                    i += 1
                else:
                    a[k] = right[j]
                    j += 1
                k += 1
                yield a[:], highlight
            while i < len(left):
                a[k] = left[i]
                i += 1
                k += 1
                yield a[:], list(range(l, r))
            while j < len(right):
                a[k] = right[j]
                j += 1
                k += 1
                yield a[:], list(range(l, r))

    yield from _merge_sort(0, n)

def quick_sort_generator(arr):
    a = arr[:]
    n = len(a)

    def _quick(l, r):
        if l < r:
            pivot = a[r]
            i = l
            for j in range(l, r):
                highlight = [j, r]
                if a[j] < pivot:
                    a[i], a[j] = a[j], a[i]
                    yield a[:], highlight
                    i += 1
            a[i], a[r] = a[r], a[i]
            yield a[:], [i, r]
            yield from _quick(l, i - 1)
            yield from _quick(i + 1, r)

    yield from _quick(0, n - 1)

# Visualizer App
class MaterialButton(tk.Button):
    def __init__(self, master=None, **kwargs):
        # 기본 스타일 정의
        default_style = {
            "bg": "#777",
            "fg": "white",
            "activebackground": "#444",
            "font": ("Roboto", 14, "bold"),
            "relief": "flat",
            "bd": 0,
            "padx": 20,
            "pady": 10,
            "cursor": "hand2"
        }

        # 사용자 정의 옵션 병합
        style = {**default_style, **kwargs}

        # 부모 클래스 초기화
        super().__init__(master, **style)

        # Hover 효과 바인딩
        self.default_bg = style["bg"]
        self.hover_bg = style.get("activebackground", "#666")
        self.bind("<Enter>", self.on_enter)
        self.bind("<Leave>", self.on_leave)

    def on_enter(self, event):
        self["bg"] = self.hover_bg

    def on_leave(self, event):
        self["bg"] = self.default_bg

class MaterialDropdown(tk.OptionMenu):
    def __init__(self, master, variable, options, command=None, **kwargs):
        self.variable = variable
        self.command = command
        self.options = options

        # 스타일 적용 가능한 옵션만 남김
        valid_keys = {"font", "relief", "bd", "highlightthickness", "indicatoron", "width"}
        button_style = {k: v for k, v in kwargs.items() if k in valid_keys}

        # 메뉴 구성
        super().__init__(master, variable, *options, command=self._on_select, **button_style)

        # 버튼 위젯 자체 스타일 수동 설정
        self.config(bg="#FFFFFF", fg="#333333", activebackground="#E0E0E0", cursor="hand2")

        # 내부 메뉴 스타일 지정
        menu = self["menu"]
        menu.config(
            bg="#FFFFFF",
            fg="#333333",
            activebackground="#999",
            font=("Roboto", 14, "bold")
        )

    def _on_select(self, value):
        if self.command:
            self.command(value)

class SortVisualizer(tk.Tk):
    def __init__(self, size=50, speed=20):
        super().__init__()
        self.title("Sorting Algorithm Visualizer")
        self.geometry("950x650")
        self.resizable(False, False)
        self.configure(bg="#ddd")

        # Algorithm mapping
        self.algorithms = {
            "Bubble Sort": bubble_sort_generator,
            "Selection Sort": selection_sort_generator,
            "Insertion Sort": insertion_sort_generator,
            "Merge Sort": merge_sort_generator,
            "Quick Sort": quick_sort_generator,
        }

        # Data and state
        self.size = size
        self.array = []
        self.generator = None
        self.step_count = 0
        self.speed = speed  # ms between auto-steps
        self.running = False

        # Canvas
        self.canvas = tk.Canvas(self, width=900, height=500, bg="white", highlightthickness=0)
        self.canvas.pack(pady=10)

        # Controls Root
        control_root = tk.Frame(self, bg="#ddd")
        control_root.pack(pady=5)

        # Algorithm selection dropdown
        self.selected_alg = tk.StringVar(value="Bubble Sort")
        dropdown = MaterialDropdown(control_root, self.selected_alg, list(self.algorithms.keys()))
        dropdown.config(width=20, padx=5, pady=5)
        dropdown.pack(side=tk.LEFT, padx=5)

        # Buttons
        self.btn_randomize = MaterialButton(control_root, text="랜덤 배열", command=self.reset_array)
        self.btn_step = MaterialButton(control_root, text="다음 단계", command=self.step)
        self.btn_run = MaterialButton(control_root, text="자동 실행", command=self.run)
        self.btn_stop = MaterialButton(control_root, text="정지", command=self.stop)
        for w in (self.btn_randomize, self.btn_step, self.btn_run, self.btn_stop):
            w.config(width=12, padx=5, pady=5, relief=tk.RAISED)
            w.pack(side=tk.LEFT, padx=5)

        # Info label
        self.info_label = tk.Label(self, text="단계: 0", font=(None, 12), bg="#ddd")
        self.info_label.pack(pady=5)

        # Initialize array
        self.reset_array()

    def reset_array(self):
        self.array = [i for i in range(1, self.size+1)]
        random.shuffle(self.array)
        self.generator = None
        self.step_count = 0
        self.running = False
        self.info_label.config(text=f"단계: {self.step_count}")
        self.draw()

    def draw(self, highlight=None):
        self.canvas.delete("all")
        highlight = highlight or []
        bar_width = 900 / len(self.array)
        max_val = max(self.array) if self.array else 1
        for i, val in enumerate(self.array):
            x0, x1 = i * bar_width, (i + 1) * bar_width - 1
            y1 = 500
            y0 = y1 - (val / max_val) * 450
            color = "#3498db" if i not in highlight else "#e74c3c"
            self.canvas.create_rectangle(x0, y0, x1, y1, fill=color, outline="")
        self.canvas.update()

    def setup_generator(self):
        alg = self.selected_alg.get()
        gen_fn = self.algorithms.get(alg, bubble_sort_generator)
        self.generator = gen_fn(self.array)

    def step(self):
        if self.generator is None:
            self.setup_generator()
        try:
            arr, highlight = next(self.generator)
            self.array = arr
            self.step_count += 1
            self.draw(highlight)
            self.info_label.config(text=f"단계: {self.step_count}")
        except StopIteration:
            self.running = False
            self.info_label.config(text=f"정렬 완료! 총 단계: {self.step_count}")

    def run(self):
        if self.generator is None:
            self.setup_generator()
        if not self.running:
            self.running = True
            self._auto_step()

    def _auto_step(self):
        if not self.running:
            return
        self.step()
        self.after(self.speed, self._auto_step)

    def stop(self):
        self.running = False

if __name__ == "__main__":
    app = SortVisualizer(size=100, speed=0) # 100
    app.mainloop()

In [5]:
# 버블 정렬
search_list = [random.randint(1, 100) for _ in range(20)]

def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]

print(search_list)
bubble_sort(search_list)
print(search_list)

[59, 8, 59, 11, 15, 17, 1, 64, 5, 28, 58, 70, 36, 35, 89, 97, 38, 13, 86, 5]
[1, 5, 5, 8, 11, 13, 15, 17, 28, 35, 36, 38, 58, 59, 59, 64, 70, 86, 89, 97]


In [6]:
# 삽입 정렬
search_list = [random.randint(1, 100) for _ in range(20)]

def insertion_sort(arr):
    for i in range(1, len(arr)):
        key = arr[i]
        j = i-1
        while j >= 0 and key < arr[j]:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key

print(search_list)
insertion_sort(search_list)
print(search_list)

[68, 93, 45, 41, 36, 86, 44, 61, 71, 53, 58, 87, 26, 43, 34, 16, 28, 57, 15, 69]
[15, 16, 26, 28, 34, 36, 41, 43, 44, 45, 53, 57, 58, 61, 68, 69, 71, 86, 87, 93]


In [ ]:
# 선택 정렬
search_list = [random.randint(1, 100) for _ in range(20)]

def selection_sort(arr):
    for i in range(len(arr)):
        min_idx = i
        for j in range(i+1, len(arr)):
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]

In [ ]:
# 머지 정렬
search_list = [random.randint(1, 100) for _ in range(20)]

def merge(arr1, arr2):
    merged = []
    i = j = 0
    while i < len(arr1) and j < len(arr2):
        if arr1[i] < arr2[j]:
            merged.append(arr1[i])
            i += 1
        else:
            merged.append(arr2[j])
            j += 1
    merged.extend(arr1[i:])
    merged.extend(arr2[j:])
    return merged

def merge_sort(arr):
    if len(arr) > 1:
        mid = len(arr) // 2
        left_merged = merge_sort(arr[:mid])
        right_merged = merge_sort(arr[mid:])
        return merge(left_merged, right_merged)
    else:
        return arr

In [7]:
# 퀵 정렬
search_list = [random.randint(1, 100) for _ in range(20)]

def quick_sort(arr):
    if len(arr) > 1:
        pivot = arr[len(arr) // 2]
    
        left = [x for x in arr if x < pivot]
        middle = [x for x in arr if x == pivot]
        right = [x for x in arr if x > pivot]

        return quick_sort(left) + middle + quick_sort(right)
    else:
        return arr

print(quick_sort(search_list))

[17, 23, 30, 32, 33, 41, 45, 48, 60, 63, 73, 73, 73, 76, 77, 78, 80, 84, 90, 94]


In [ ]:
# 순차 탐색
search_list = [random.randint(1, 100) for _ in range(20)]

def sequential_search(target, search_list):
    for i in range(len(search_list)):
        if search_list[i] == target:
            return i
    return -1

In [ ]:
# 이진 탐색
search_list = [random.randint(1, 100) for _ in range(20)]

def binary_search(target, search_list):
    left, right = 0, len(search_list) - 1
    while left <= right:
        mid = (left + right) // 2
        if search_list[mid] == target:
            return mid
        elif search_list[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

In [18]:
# 머지 소트를 queue로 구현

arr1 = [random.randint(1, 100) for _ in range(20)]

def merge(arr1, arr2):
    merged = []
    i = j = 0
    while i < len(arr1) and j < len(arr2):
        if arr1[i] < arr2[j]:
            merged.append(arr1[i])
            i += 1
        else:
            merged.append(arr2[j])
            j += 1
    merged.extend(arr1[i:])
    merged.extend(arr2[j:])
    return merged

def merge_sort_queue(arr):
    queue = [[x] for x in arr]
    
    while len(queue) > 1:
        l1 = queue.pop(0)
        l2 = queue.pop(0)
        queue.append(merge(l1, l2))
    
    return queue[0]

print(arr1)
print(merge_sort_queue(arr1))

[99, 59, 43, 31, 83, 75, 65, 96, 31, 17, 56, 80, 77, 76, 68, 10, 5, 4, 60, 4]
[4, 4, 5, 10, 17, 31, 31, 43, 56, 59, 60, 65, 68, 75, 76, 77, 80, 83, 96, 99]


In [19]:
# 퀵 소트를 stack으로 구현

arr2 = [random.randint(1, 100) for _ in range(20)]

def quick_sort_stack(arr):
    stack = [(0, len(arr) - 1)]
    while stack:
        left, right = stack.pop()
        if left < right:
            pivot = arr[(left + right) // 2]
            i = left
            j = right
            while i <= j:
                while arr[i] < pivot:
                    i += 1
                while arr[j] > pivot:
                    j -= 1
                if i <= j:
                    arr[i], arr[j] = arr[j], arr[i]
                    i += 1
                    j -= 1
            stack.append((left, j))
            stack.append((i, right))
    return arr

print(arr1)
print(quick_sort_stack(arr2))

[99, 59, 43, 31, 83, 75, 65, 96, 31, 17, 56, 80, 77, 76, 68, 10, 5, 4, 60, 4]
[6, 9, 19, 20, 21, 38, 50, 50, 51, 53, 55, 61, 62, 68, 77, 92, 93, 94, 94, 99]
